# Root-Causal Graph demonstration

This post-hoc demo uses an already generated BGL graph tensor and an already trained OCDiGCN checkpoint. It does not parse logs, regenerate graphs, create splits, or retrain the model.

In [ ]:
from pathlib import Path
import os
import sys

import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
from root_causal import analyze_root_causes, draw_root_causal_graph

# Point these at existing local artifacts; no artifact is downloaded or regenerated.
DATASET_PATH = Path(os.environ.get('BGL_ROOT_CAUSAL_DATASET', REPO_ROOT / 'bgl_paper_digcn_graphs.pt'))
CHECKPOINT_PATH = Path(os.environ.get('BGL_ROOT_CAUSAL_CHECKPOINT', REPO_ROOT / 'ocdigcn_bgl.pt'))
assert DATASET_PATH.exists(), f'Missing existing graph tensor: {DATASET_PATH}'
assert CHECKPOINT_PATH.exists(), f'Missing trained checkpoint: {CHECKPOINT_PATH}'


In [ ]:
from DataLoader import DiGCN

graphs = torch.load(DATASET_PATH, weights_only=False)
if isinstance(graphs, dict):
    graphs = graphs['graphs']
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
state_dict = checkpoint.get('model_state_dict', checkpoint)
center = checkpoint.get('center')
assert center is not None, 'Checkpoint must contain the trained SVDD center.'

model = DiGCN(nfeat=graphs[0].x.size(1), nhid=center.numel(), nlayer=1, bias=False)
model.load_state_dict(state_dict)
model.eval()


In [ ]:
# Score every already prepared graph and choose the highest-scoring anomaly.
scores, node_embeddings = [], []
with torch.no_grad():
    for graph in graphs:
        graph.batch = torch.zeros(graph.num_nodes, dtype=torch.long)
        embeddings = model(graph)[0]
        graph_embedding = embeddings.mean(dim=0)
        scores.append(torch.sum((graph_embedding - center) ** 2).item())
        node_embeddings.append(embeddings)
anomalous_index = max(range(len(scores)), key=scores.__getitem__)
anomalous_graph = graphs[anomalous_index]
embeddings = node_embeddings[anomalous_index]
# Transparent node explanation: distance from the graph representation.
node_importance = torch.linalg.vector_norm(embeddings - embeddings.mean(dim=0), dim=1).cpu().numpy()
print(f'Anomalous graph index: {anomalous_index}; OCDiGCN score: {scores[anomalous_index]:.4f}')


In [ ]:
result = analyze_root_causes(anomalous_graph, node_importance, top_k=10, max_hops=4)
print('Ranked root causes:')
for node, confidence in result.ranked_roots[:5]:
    print(f'  {node}: {confidence:.2f}')
print('\nCausal chains:')
for chain in result.causal_chains:
    print('  ' + ' -> '.join(map(str, chain.nodes)) + f' ({chain.confidence:.2f})')
output = draw_root_causal_graph(result, REPO_ROOT / 'root_causal_graph.png')
output
